# Thuiszorg Route Algoritme — Heerlen

## Structuur
1. Installaties & imports
2. Data laden uit CSV
3. Geocoding: adressen naar GPS coordinaten
4. Matching: medewerker en client constraints
5. Afstandsmatrix berekenen
6. OR-Tools VRP oplossen
7. Resultaat visualiseren op kaart

## 1. Installaties & Imports

In [ ]:
# Uncomment om te installeren indien nodig
# %pip install ortools geopy folium pandas
# %pip install osmnx network

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for network: filename=network-0.1-py3-none-any.whl size=3182 sha256=d9cf11e8cc3af31a1409610464abbaee8fc05b0e7f9760ea5840ef2678566b70
  Stored in directory: c:\users\noudr\appdata\local\pip\cache\wheels\e7\5a\7a\7f15bea66afb5505b9d10cc7bd8964cb77f0ce736df5b104c8
Successfully built network
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import math
import time
import folium
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
from IPython.display import display
import osmnx as ox
import networkx as nx
import os
import pickle

print('Imports geslaagd')

Imports geslaagd


## 2. Data laden uit CSV
Pas de paden aan naar jouw mapstructuur.

In [4]:
# Paden aanpassen indien nodig
EMPLOYEES_CSV = '../backend/employees.csv'
CLIENTEN_CSV  = '../backend/clienten.csv'

# Medewerkers laden
df_mw = pd.read_csv(EMPLOYEES_CSV)
print(f'{len(df_mw)} medewerkers geladen')
print(df_mw[['name','address','time_window_start','time_window_end','dogs','cats','smokes']].to_string())

20 medewerkers geladen
            name                                   address time_window_start time_window_end  dogs  cats  smokes
0         Vickie           Kanaalstraat 11 6418 NI Heerlen             07:00           18:00     0    -1   False
1           Anna               Lindelaan 3 6414 CW Heerlen             07:00           18:00    -1     2    True
2         Lauren        Oude Lindestraat 4 6411 YU Heerlen             07:00           18:00    -1     0   False
3    Juan Manuel  Deken Nicolaijestraat 18 6411 HC Heerlen             07:00           18:00    -1     1   False
4        Paulina             Julianaweg 50 6413 XK Heerlen             07:00           18:00    -1     2   False
5          Zlata          Tollensstraat 31 6416 GK Heerlen             07:00           18:00     0     1   False
6          Mason         Raadhuisstraat 20 6411 HW Heerlen             07:00           18:00    -1    -1    True
7      Hans-Theo                 Bongerd 7 6411 CZ Heerlen             07

In [5]:
# Clienten laden
df_cl = pd.read_csv(CLIENTEN_CSV)

# Volledig adres samenvoegen
df_cl['address'] = df_cl['Straat'] + ' ' + df_cl['Postcode'].astype(str) + ' ' + df_cl['Stad']

# Constraints parsen uit de Opmerkingen kolom
df_cl['heeft_hond'] = df_cl['Opmerkingen'].fillna('').str.contains('Hond', case=False)
df_cl['heeft_kat']  = df_cl['Opmerkingen'].fillna('').str.contains('Kat',  case=False)
df_cl['rookt']      = df_cl['Opmerkingen'].fillna('').str.contains('Rookt', case=False)

print(f'{len(df_cl)} clienten geladen')
print(df_cl[['Naam','address','Duur (min)','Tijdvensters','heeft_hond','heeft_kat','rookt']].to_string())

100 clienten geladen
                 Naam                                          address  Duur (min) Tijdvensters  heeft_hond  heeft_kat  rookt
0         Mw. Nijssen                  Bethlehemstraat 24 6418 Heerlen         120  08:00-12:00       False       True  False
1       Dhr. Lucassen                      Trompstraat 30 6412 Heerlen          60  08:00-12:00       False      False  False
2          Dhr. Bours                   Nazarethstraat 40 6418 Heerlen          60  08:00-12:00       False      False   True
3        Dhr. Hanssen                     Bradleystraat 4 6418 Heerlen         180  12:00-18:00        True      False  False
4         Dhr. Strous                   Hambeukerboord 40 6418 Heerlen         120  08:00-12:00       False       True   True
5         Mw. Dormans  Laan van Hövell tot Westerflier 29 6411 Heerlen          90  08:00-12:00       False       True  False
6      Mw. Steinbusch                 Coriovallumstraat 3 6411 Heerlen          90  08:00-12:00  

## 3. Geocoding: adressen naar GPS coordinaten

We gebruiken Nominatim (OpenStreetMap) om elk adres om te zetten naar lat/lon.

Let op: dit duurt even vanwege de rate limit van 1 request per seconde.
Resultaten worden opgeslagen zodat je dit maar eenmalig hoeft te doen.

In [6]:
CACHE_DIR  = '../data/osm_cache'
CACHE_AUTO = os.path.join(CACHE_DIR, 'heerlen_drive.pkl')
CACHE_FIETS = os.path.join(CACHE_DIR, 'heerlen_bike.pkl')

os.makedirs(CACHE_DIR, exist_ok=True)

def laad_of_download_kaart(cache_pad, netwerk_type, plaatsnaam='Heerlen, Netherlands'):
    """Laadt kaart uit cache of download via OSMnx."""
    if os.path.exists(cache_pad):
        print(f'  Kaart geladen uit cache: {cache_pad}')
        with open(cache_pad, 'rb') as f:
            return pickle.load(f)
    else:
        print(f'  Downloading {netwerk_type} kaart van {plaatsnaam}...')
        G = ox.graph_from_place(plaatsnaam, network_type=netwerk_type)
        # Voeg reistijden toe aan edges
        if netwerk_type == 'drive':
            G = ox.add_edge_speeds(G)
        G = ox.add_edge_travel_times(G)
        with open(cache_pad, 'wb') as f:
            pickle.dump(G, f)
        print(f'  Gecached naar: {cache_pad}')
        return G

print('Auto netwerk laden...')
G_auto = laad_of_download_kaart(CACHE_AUTO, 'drive')
print(f'  Nodes: {len(G_auto.nodes):,}  |  Edges: {len(G_auto.edges):,}')

print('Fiets netwerk laden...')
G_fiets = laad_of_download_kaart(CACHE_FIETS, 'bike')
print(f'  Nodes: {len(G_fiets.nodes):,}  |  Edges: {len(G_fiets.edges):,}')

print('\nKaarten klaar!')

Auto netwerk laden...
  Kaart geladen uit cache: ../data/osm_cache\heerlen_drive.pkl
  Nodes: 3,617  |  Edges: 8,584
Fiets netwerk laden...


KeyError: "All edges must have 'length' and 'speed_kph' attributes."

In [8]:
def haversine(coord1, coord2):
    """Fallback: luchtlijn afstand in meters."""
    import math
    R = 6371000
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def maak_osm_afstandsmatrix(G, nodes, alle_coords, gewicht='length'):
    """
    Bouwt NxN afstandsmatrix via kortste paden in OSMnx.

    Parameters:
        G         : OSMnx graaf
        nodes     : lijst van node IDs per punt
        alle_coords: originele coordinaten (voor fallback)
        gewicht   : 'length' voor meters, 'travel_time' voor seconden

    Returns:
        NxN matrix met afstanden als integers
    """
    n = len(nodes)
    matrix = [[0] * n for _ in range(n)]
    niet_bereikbaar = 0

    for i in range(n):
        # Bereken kortste paden vanuit node i naar alle andere nodes
        try:
            lengtes = nx.single_source_dijkstra_path_length(
                G, nodes[i], weight=gewicht
            )
        except nx.NodeNotFound:
            lengtes = {}

        for j in range(n):
            if i == j:
                continue
            if nodes[j] in lengtes:
                matrix[i][j] = int(lengtes[nodes[j]])
            else:
                # Fallback: haversine * 1.4 (omrijfactor)
                matrix[i][j] = int(haversine(alle_coords[i], alle_coords[j]) * 1.4)
                niet_bereikbaar += 1

    if niet_bereikbaar > 0:
        print(f'  Let op: {niet_bereikbaar} paren niet bereikbaar, fallback gebruikt')

    return matrix


# Auto matrix (afstand in meters)
print('Auto afstandsmatrix berekenen...')
matrix_auto = maak_osm_afstandsmatrix(G_auto, nodes_auto, alle_coords, gewicht='length')
print(f'  Klaar! Voorbeeld: {df_mw_ok.iloc[0]["name"]} -> {df_cl_ok.iloc[0]["Naam"]}: {matrix_auto[0][n_mw]:.0f} m')

# Fiets matrix (afstand in meters)
print('Fiets afstandsmatrix berekenen...')
matrix_fiets = maak_osm_afstandsmatrix(G_fiets, nodes_fiets, alle_coords, gewicht='length')
print(f'  Klaar! Voorbeeld: {df_mw_ok.iloc[0]["name"]} -> {df_cl_ok.iloc[0]["Naam"]}: {matrix_fiets[0][n_mw]:.0f} m')

# Haversine matrix voor vergelijking
print('Luchtlijn matrix berekenen (vergelijking)...')
matrix_luchtlijn = [[0]*len(alle_coords) for _ in range(len(alle_coords))]
for i in range(len(alle_coords)):
    for j in range(len(alle_coords)):
        if i != j:
            matrix_luchtlijn[i][j] = int(haversine(alle_coords[i], alle_coords[j]))
print('  Klaar!')

Auto afstandsmatrix berekenen...


NameError: name 'nodes_auto' is not defined

In [9]:
import matplotlib.pyplot as plt
import numpy as np

# Verzamel alle paar-afstanden (alleen mw -> client)
luchtlijn_afstanden = []
auto_afstanden      = []
fiets_afstanden     = []
labels              = []

for mw_idx in range(n_mw):
    for cl_idx in range(n_cl):
        node_cl = n_mw + cl_idx
        luchtlijn_afstanden.append(matrix_luchtlijn[mw_idx][node_cl] / 1000)
        auto_afstanden.append(matrix_auto[mw_idx][node_cl] / 1000)
        fiets_afstanden.append(matrix_fiets[mw_idx][node_cl] / 1000)
        labels.append(f"{df_mw_ok.iloc[mw_idx]['name']} -> {df_cl_ok.iloc[cl_idx]['Naam']}")

# Omrijfactoren
factor_auto  = np.mean([a/l for a, l in zip(auto_afstanden,  luchtlijn_afstanden) if l > 0])
factor_fiets = np.mean([f/l for f, l in zip(fiets_afstanden, luchtlijn_afstanden) if l > 0])

print('=' * 50)
print('VERGELIJKING AFSTANDSMETHODEN')
print('=' * 50)
print(f'Gemiddelde afstand luchtlijn : {np.mean(luchtlijn_afstanden):.2f} km')
print(f'Gemiddelde afstand auto      : {np.mean(auto_afstanden):.2f} km')
print(f'Gemiddelde afstand fiets     : {np.mean(fiets_afstanden):.2f} km')
print()
print(f'Omrijfactor auto  : {factor_auto:.2f}x  (auto is gem. {(factor_auto-1)*100:.0f}% langer dan luchtlijn)')
print(f'Omrijfactor fiets : {factor_fiets:.2f}x  (fiets is gem. {(factor_fiets-1)*100:.0f}% langer dan luchtlijn)')

# Grafiek
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram van afstanden
axes[0].hist(luchtlijn_afstanden, bins=30, alpha=0.6, label='Luchtlijn', color='gray')
axes[0].hist(auto_afstanden,      bins=30, alpha=0.6, label='Auto',      color='#2563eb')
axes[0].hist(fiets_afstanden,     bins=30, alpha=0.6, label='Fiets',     color='#16a34a')
axes[0].set_xlabel('Afstand (km)')
axes[0].set_ylabel('Aantal paren')
axes[0].set_title('Verdeling afstanden per methode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: luchtlijn vs auto
axes[1].scatter(luchtlijn_afstanden, auto_afstanden,  alpha=0.3, s=15, color='#2563eb', label='Auto')
axes[1].scatter(luchtlijn_afstanden, fiets_afstanden, alpha=0.3, s=15, color='#16a34a', label='Fiets')
max_val = max(max(luchtlijn_afstanden), max(auto_afstanden), max(fiets_afstanden))
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='1:1 lijn')
axes[1].set_xlabel('Luchtlijn (km)')
axes[1].set_ylabel('Werkelijke afstand (km)')
axes[1].set_title('Luchtlijn vs werkelijke afstand')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Vergelijking afstandsmethoden — Heerlen', fontweight='bold')
plt.tight_layout()
plt.savefig('../output/afstand_vergelijking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafiek opgeslagen in output/afstand_vergelijking.png')

NameError: name 'n_mw' is not defined

In [10]:
# --- KEUZE: 'auto' of 'fiets' ---
TRANSPORT_MODUS = 'auto'
# --------------------------------

if TRANSPORT_MODUS == 'auto':
    afstand_matrix = matrix_auto
    print('Modus: AUTO')
elif TRANSPORT_MODUS == 'fiets':
    afstand_matrix = matrix_fiets
    print('Modus: FIETS')
else:
    raise ValueError(f'Onbekende modus: {TRANSPORT_MODUS}. Kies auto of fiets.')

print(f'Afstandsmatrix klaar voor OR-Tools: {len(afstand_matrix)}x{len(afstand_matrix[0])}')
print('Ga nu verder met cel 6: OR-Tools VRP oplossen')

NameError: name 'matrix_auto' is not defined

## 4. Matching: medewerker en client constraints

Regels vanuit de data:
- `dogs = -1`  -> medewerker allergisch voor honden -> mag NIET naar client met hond
- `cats = -1`  -> medewerker allergisch voor katten -> mag NIET naar client met kat
- `smokes = false` -> medewerker rookt niet -> bij voorkeur niet naar rokende client

Positieve waarden (bijv. dogs = 2) betekenen dat de medewerker prima met honden overweg kan.

In [11]:
def mag_koppelen(mw_rij, cl_rij, strikt_roken=False):
    """
    Geeft True terug als deze medewerker naar deze client kan.
    
    Parameters:
        strikt_roken : als True, gaan niet-rokers nooit naar rokende clienten
    """
    # Hond allergie: dogs = -1 betekent allergisch
    if int(mw_rij['dogs']) == -1 and cl_rij['heeft_hond']:
        return False

    # Kat allergie: cats = -1 betekent allergisch
    if int(mw_rij['cats']) == -1 and cl_rij['heeft_kat']:
        return False

    # Roken (optioneel strikt)
    if strikt_roken and str(mw_rij['smokes']).lower() == 'false' and cl_rij['rookt']:
        return False

    return True


# Bouw matching matrix
match_matrix = []
for _, mw in df_mw_ok.iterrows():
    rij = [mag_koppelen(mw, cl) for _, cl in df_cl_ok.iterrows()]
    match_matrix.append(rij)

# Samenvatting
print('Matching samenvatting per medewerker:')
for mw_idx, mw in df_mw_ok.iterrows():
    toegestaan = sum(match_matrix[mw_idx])
    print(f'  {mw["name"]:15s} -> {toegestaan}/{len(df_cl_ok)} clienten toegestaan')

# Waarschuwingen
print()
for cl_idx, cl in df_cl_ok.iterrows():
    mogelijke_mw = sum(match_matrix[mw_idx][cl_idx] for mw_idx in range(len(df_mw_ok)))
    if mogelijke_mw == 0:
        print(f'WAARSCHUWING: {cl["Naam"]} heeft GEEN geschikte medewerker!')
    elif mogelijke_mw <= 2:
        print(f'Let op: {cl["Naam"]} heeft slechts {mogelijke_mw} geschikte medewerker(s)')

NameError: name 'df_mw_ok' is not defined

## 5. Afstandsmatrix berekenen

Haversine afstand (echte aardbol-afstand in meters) tussen alle punten.

Later te vervangen door echte rijafstanden via OSMnx op de wegenkaart van Heerlen.

In [12]:
def haversine(coord1, coord2):
    """Berekent afstand in meters tussen twee GPS coordinaten."""
    R = 6371000
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def maak_afstandsmatrix(df_mw, df_cl):
    """
    Bouwt NxN matrix in meters.
    Index 0 t/m (n_mw-1) = medewerker depots
    Index n_mw t/m einde = clienten
    """
    alle_coords = list(df_mw['coords']) + list(df_cl['coords'])
    n = len(alle_coords)
    matrix = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = int(haversine(alle_coords[i], alle_coords[j]))
    return matrix, alle_coords


afstand_matrix, alle_coords = maak_afstandsmatrix(df_mw_ok, df_cl_ok)
n_mw = len(df_mw_ok)
n_cl = len(df_cl_ok)

print(f'Afstandsmatrix klaar: {len(alle_coords)}x{len(alle_coords)}')
print(f'  {n_mw} medewerkers + {n_cl} clienten')
print(f'  Voorbeeld: {df_mw_ok.iloc[0]["name"]} -> {df_cl_ok.iloc[0]["Naam"]}: {afstand_matrix[0][n_mw]:.0f} meter')

NameError: name 'df_mw_ok' is not defined

## 6. OR-Tools VRP oplossen

- Elke medewerker = voertuig met eigen depot (woonadres)
- Elke client = stop die bezocht moet worden
- Matching constraints worden afgedwongen via VehicleVar
- Doel: minimaliseer totale reisafstand met gebalanceerde werkdruk

In [ ]:
def los_vrp_op(afstand_matrix, match_matrix, n_mw, n_cl, max_tijd=30):
    """
    Lost het Vehicle Routing Problem op met OR-Tools.

    Parameters:
        afstand_matrix : NxN matrix met afstanden in meters
        match_matrix   : n_mw x n_cl boolean matrix (True = toegestaan)
        n_mw           : aantal medewerkers
        n_cl           : aantal clienten
        max_tijd       : maximale rekentijd in seconden

    Returns:
        routes : dict {mw_idx: [lijst van cl_idx]}
        status : string
    """
    n_nodes = n_mw + n_cl
    depots  = list(range(n_mw))

    manager = pywrapcp.RoutingIndexManager(n_nodes, n_mw, depots, depots)
    routing = pywrapcp.RoutingModel(manager)

    # Afstandsfunctie registreren
    def afstand_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node   = manager.IndexToNode(to_index)
        return afstand_matrix[from_node][to_node]

    transit_idx = routing.RegisterTransitCallback(afstand_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

    # Afstandsdimensie voor balancering van werkdruk
    routing.AddDimension(transit_idx, 0, 10_000_000, True, 'Afstand')
    routing.GetDimensionOrDie('Afstand').SetGlobalSpanCostCoefficient(100)

    # Matching constraints via VehicleVar
    # Elke client krijgt alleen de medewerkers toegewezen die mogen
    for cl_idx in range(n_cl):
        cl_node = n_mw + cl_idx
        cl_routing_idx = manager.NodeToIndex(cl_node)
        toegestane_mw = [mw_idx for mw_idx in range(n_mw) if match_matrix[mw_idx][cl_idx]]
        if toegestane_mw:
            routing.VehicleVar(cl_routing_idx).SetValues(toegestane_mw)
        # Als geen enkele mw toegestaan is, laat OR-Tools zelf kiezen (fallback)

    # Zoekstrategie
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    params.time_limit.seconds = max_tijd

    print(f'OR-Tools bezig... (max {max_tijd} seconden)')
    oplossing = routing.SolveWithParameters(params)

    if not oplossing:
        return {}, 'GEEN OPLOSSING'

    # Routes uitlezen
    routes = {}
    for mw_idx in range(n_mw):
        index = routing.Start(mw_idx)
        route = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            if node >= n_mw:
                route.append(node - n_mw)
            index = oplossing.Value(routing.NextVar(index))
        routes[mw_idx] = route

    status_map = {1: 'OPTIMAAL', 2: 'GOED GENOEG'}
    status = status_map.get(routing.status(), f'STATUS CODE {routing.status()}')
    return routes, status


routes, status = los_vrp_op(afstand_matrix, match_matrix, n_mw, n_cl, max_tijd=15)

print(f'\nResultaat: {status}')
print('\n--- Routes per medewerker ---')
for mw_idx, route in routes.items():
    mw_naam = df_mw_ok.iloc[mw_idx]['name']
    stops   = [df_cl_ok.iloc[i]['Naam'] for i in route]
    zorg    = sum(df_cl_ok.iloc[i]['Duur (min)'] for i in route)
    print(f'  {mw_naam:15s} -> {len(route)} stops, {zorg} min: {" -> ".join(stops) if stops else "(geen stops)"}')

## 7. Visualisatie op kaart van Heerlen

Interactieve kaart via Folium. Klik op een marker voor details.

In [ ]:
KLEUREN = [
    'blue', 'green', 'purple', 'orange', 'red',
    'darkblue', 'darkgreen', 'cadetblue', 'darkpurple',
    'pink', 'lightblue', 'lightgreen', 'gray', 'black',
    'lightgray', 'beige', 'lightred', 'darkred'
]

def visualiseer_op_kaart(df_mw, df_cl, routes):
    """Toont alle routes op een interactieve Folium kaart van Heerlen."""

    kaart = folium.Map(
        location=[50.8884, 5.9799],  # centrum Heerlen
        zoom_start=13,
        tiles='OpenStreetMap'
    )

    for mw_idx, route in routes.items():
        mw    = df_mw.iloc[mw_idx]
        kleur = KLEUREN[mw_idx % len(KLEUREN)]

        # Medewerker depot marker
        folium.Marker(
            location=mw['coords'],
            popup=folium.Popup(
                f"<b>{mw['name']}</b><br>"
                f"{mw['address']}<br>"
                f"Werktijden: {mw['time_window_start']} - {mw['time_window_end']}<br>"
                f"Stops vandaag: {len(route)}",
                max_width=250
            ),
            tooltip=mw['name'],
            icon=folium.Icon(color=kleur, icon='home', prefix='fa')
        ).add_to(kaart)

        if not route:
            continue

        # Route lijn: depot -> clienten -> depot
        route_coords = [mw['coords']]
        for stap, cl_idx in enumerate(route):
            cl = df_cl.iloc[cl_idx]
            route_coords.append(cl['coords'])

            # Client marker
            folium.CircleMarker(
                location=cl['coords'],
                radius=9,
                color=kleur,
                fill=True,
                fill_color=kleur,
                fill_opacity=0.85,
                popup=folium.Popup(
                    f"<b>{cl['Naam']}</b><br>"
                    f"Stop {stap + 1} van {mw['name']}<br>"
                    f"Zorg: {cl['Type Zorg']}<br>"
                    f"Duur: {cl['Duur (min)']} min<br>"
                    f"Tijdvenster: {cl['Tijdvensters']}<br>"
                    f"Opmerkingen: {cl['Opmerkingen'] if pd.notna(cl['Opmerkingen']) else '-'}",
                    max_width=280
                ),
                tooltip=f"{stap + 1}. {cl['Naam']}"
            ).add_to(kaart)

        route_coords.append(mw['coords'])  # terug naar depot

        folium.PolyLine(
            locations=route_coords,
            color=kleur,
            weight=3,
            opacity=0.7,
            tooltip=f"{mw['name']} ({len(route)} stops)"
        ).add_to(kaart)

    return kaart


kaart = visualiseer_op_kaart(df_mw_ok, df_cl_ok, routes)
display(kaart)

## 8. Samenvatting

In [ ]:
print('=' * 60)
print('PLANNING SAMENVATTING')
print('=' * 60)
print(f'Status           : {status}')
print(f'Medewerkers      : {n_mw}')
print(f'Clienten         : {n_cl}')
print()

totaal_afstand  = 0
totaal_zorg     = 0
ingeplande_cl   = set(cl_idx for route in routes.values() for cl_idx in route)

for mw_idx, route in routes.items():
    mw       = df_mw_ok.iloc[mw_idx]
    stops    = [df_cl_ok.iloc[i]['Naam'] for i in route]
    zorg_min = sum(df_cl_ok.iloc[i]['Duur (min)'] for i in route)
    totaal_zorg += zorg_min

    punten = [mw['coords']] + [df_cl_ok.iloc[i]['coords'] for i in route] + [mw['coords']]
    reis   = sum(haversine(punten[i], punten[i+1]) for i in range(len(punten)-1))
    totaal_afstand += reis

    print(f"{mw['name']}:")
    print(f"  Stops       : {len(route)} clienten")
    print(f"  Zorgtijd    : {zorg_min} min")
    print(f"  Reisafstand : {reis / 1000:.1f} km")
    if stops:
        print(f"  Volgorde    : {' -> '.join(stops)}")
    print()

niet_ingepland = [df_cl_ok.iloc[i]['Naam'] for i in range(n_cl) if i not in ingeplande_cl]

print(f'Totale reisafstand : {totaal_afstand / 1000:.1f} km')
print(f'Totale zorgtijd    : {totaal_zorg} min ({totaal_zorg / 60:.1f} uur)')

if niet_ingepland:
    print(f'\nNiet ingepland ({len(niet_ingepland)}):')
    for naam in niet_ingepland:
        print(f'  - {naam}')
else:
    print('\nAlle clienten zijn ingepland!')

---
## Roadmap

| Stap | Wat | Status |
|------|-----|--------|
| 1 | CSV data laden + matching constraints | Klaar |
| 2 | Geocoding: adressen naar GPS | Klaar |
| 3 | OR-Tools VRP met matching | Klaar |
| 4 | Visualisatie op Folium kaart | Klaar |
| 5 | OSMnx: echte rijafstanden Heerlen | Volgende stap |
| 6 | Tijdvenster constraints in VRP | Daarna |
| 7 | Flask API koppeling met dashboard | Daarna |
| 8 | Leaflet kaart in dashboard | Daarna |